# Workbook 1 · ¿Cumplirá esta mezcla de concreto la resistencia exigida?

En este cuaderno vas a recorrer, paso a paso, el camino completo **de los datos a una decisión de ingeniería**:

1. Cargar y explorar datos reales de laboratorio
2. Entrenar dos modelos de Machine Learning (ML)
3. Medir qué tan bien predicen
4. Usar el modelo para decidir si una mezcla se **aprueba** o se **rechaza**

No necesitas saber programar. Casi todo el código ya está escrito: tu trabajo es **ejecutarlo, leer qué hace, cambiar valores y observar qué pasa**.


## 0 · Cómo usar este cuaderno

- Un cuaderno está hecho de **celdas**. Las de texto, como esta, explican; las de código se ejecutan.
- Para ejecutar una celda de código: haz clic en ella y presiona **Shift + Enter**, o el botón ▶.
- **Ejecuta las celdas en orden, de arriba hacia abajo.** Si algo falla, casi siempre es porque se saltó una celda.
- Si te pierdes: menú **Entorno de ejecución → Reiniciar y ejecutar todo**.
- Un error en rojo **no rompe nada**. Lee la última línea, suele decir qué pasó. Pide ayuda sin pena.

## 1 · Python en 5 minutos

Python es un lenguaje para dar instrucciones a la computadora. Para hoy solo necesitas tres ideas: **variables**, **operaciones** y **listas**.

In [ ]:
# Una variable guarda un valor con un nombre. Todo lo que sigue a # es un comentario: Python lo ignora.
temperatura = 25
print("La temperatura es", temperatura, "°C")

In [ ]:
# Operaciones: Python es una calculadora con memoria
cemento = 300   # kg por metro cúbico
agua = 180      # kg por metro cúbico

relacion_agua_cemento = agua / cemento
print("Relación agua/cemento:", relacion_agua_cemento)

In [ ]:
# Una lista guarda varios valores
resistencias = [25.0, 32.5, 41.2, 38.0]   # MPa de cuatro probetas

print("Promedio:", sum(resistencias) / len(resistencias))
print("Máxima:", max(resistencias))
print("¿Cuántas probetas?", len(resistencias))

> **Tu turno (1 min):** en la celda de las operaciones cambia `agua = 180` por `agua = 150` y vuelve a ejecutarla. ¿Qué pasó con la relación agua/cemento? En concreto, menos agua suele significar más resistencia: guarda esa intuición, el modelo la va a redescubrir.

## 2 · Cargar las librerías y los datos

Una **librería** es una caja de herramientas ya hecha. Usaremos cuatro:

| Librería | Para qué sirve |
|---|---|
| `pandas` | Manejar tablas de datos (como Excel, pero con código) |
| `numpy` | Cálculos numéricos |
| `matplotlib` | Hacer gráficas |
| `scikit-learn` | Los modelos de Machine Learning y sus métricas |

In [ ]:
import io
import os
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Librerías listas")

### Los datos

Usaremos el conjunto **Concrete Compressive Strength** (Yeh, 1998) del repositorio UCI: **1,030 mezclas de concreto** ensayadas en laboratorio. Cada fila es una mezcla; las columnas son sus ingredientes (en kg por m³ de concreto), la edad de la probeta al ensayo (en días) y la **resistencia a compresión (MPa)**, que es lo que queremos predecir.

- Página del conjunto: https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength
- Licencia: CC BY 4.0 · DOI: 10.24432/C5PK67

In [ ]:
URL_CONCRETO = "https://archive.ics.uci.edu/static/public/165/concrete+compressive+strength.zip"

COLUMNAS = ["cemento", "escoria", "ceniza_volante", "agua", "superplastificante",
            "agregado_grueso", "agregado_fino", "edad_dias", "resistencia_MPa"]


def cargar_concreto():
    print("Descargando desde UCI...")
    with urllib.request.urlopen(URL_CONCRETO, timeout=60) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    archivo = [n for n in z.namelist() if n.lower().endswith((".xls", ".xlsx"))][0]
    return pd.read_excel(z.open(archivo))


df = cargar_concreto()
assert df.shape[1] == 9, "Se esperaban 9 columnas"
df.columns = COLUMNAS            # nombres cortos y en español
print("Listo:", df.shape[0], "mezclas y", df.shape[1], "columnas")

> **¿Falló la descarga?** (1) Si dice `xlrd`, ejecuta en una celda nueva `!pip -q install xlrd` y repite. (2) Si no hay internet, el instructor te pasará el archivo `concreto.csv`: súbelo con el ícono de carpeta 📁 de la izquierda y vuelve a ejecutar la celda anterior.

## 3 · Explorar los datos

Antes de modelar, **siempre mira los datos**. Un modelo con datos mal entendidos da respuestas bonitas pero equivocadas.

In [ ]:
df.head()        # las primeras 5 filas

In [ ]:
print("Filas y columnas:", df.shape)
print()
print("Valores faltantes por columna:")
print(df.isna().sum())

In [ ]:
df.describe().round(1)   # resumen estadístico: mínimo, máximo, promedio...

**Responde en voz alta con tu pareja:**

1. ¿Cuál es la resistencia mínima, la promedio y la máxima?
2. ¿Entre qué valores varía la edad de las probetas?¿Hay mezclas ensayadas a 1 día y a 1 año?
3. ¿Hay datos faltantes? (Pista: la tabla de arriba debería mostrar solo ceros.)

Guarda estos rangos en la memoria: más adelante veremos por qué **importan mucho** cuando el modelo predice una mezcla nueva.

## 4 · Graficar: ¿qué se relaciona con la resistencia?

In [ ]:
plt.hist(df["resistencia_MPa"], bins=25, color="#4C72B0", edgecolor="white")
plt.xlabel("Resistencia a compresión (MPa)")
plt.ylabel("Número de mezclas")
plt.title("¿Qué resistencias hay en nuestros datos?")
plt.show()

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
for eje, col in zip(ejes, ["cemento", "agua", "edad_dias"]):
    eje.scatter(df[col], df["resistencia_MPa"], s=8, alpha=0.5)
    eje.set_xlabel(col)
    eje.set_ylabel("Resistencia (MPa)")
    eje.set_title("Resistencia vs " + col)
plt.tight_layout()
plt.show()

In [ ]:
# Correlación: qué tan de la mano van dos variables (de -1 a +1)
correlaciones = df.corr()["resistencia_MPa"].drop("resistencia_MPa").sort_values()
correlaciones.plot(kind="barh", color="#4C72B0")
plt.xlabel("Correlación con la resistencia")
plt.title("Variables relacionadas con la resistencia")
plt.show()
print(correlaciones.round(2))

> **Cuidado:** que dos variables se muevan juntas (correlación) **no prueba que una cause la otra**. Es una pista para investigar, no una conclusión.

**Escribe tu hipótesis:** ¿qué dos variables crees que más ayudarán a predecir la resistencia? ¿Por qué?

## 5 · Separar datos de entrenamiento y de prueba

Para saber si un modelo **realmente aprendió** (y no solo memorizó), lo evaluamos con datos que nunca vio. Es como un examen: no te evalúan con los mismos ejercicios que practicaste.

- **Entrenamiento (80 %)**: el modelo aprende de estas mezclas.
- **Prueba (20 %)**: las guardamos bajo llave y solo las usamos para calificar.

`X` son las variables de entrada (ingredientes y edad) y `y` es lo que queremos predecir (resistencia).

In [ ]:
X = df.drop(columns="resistencia_MPa")
y = df["resistencia_MPa"]

X_ent, X_prueba, y_ent, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42)

print("Mezclas para entrenar:", len(X_ent))
print("Mezclas para probar:  ", len(X_prueba))

## 6 · Primer modelo: regresión lineal

La **regresión lineal** busca una fórmula del tipo:

> resistencia ≈ *a* · cemento + *b* · agua + *c* · edad + … + constante

Es simple, rápida e interpretable. Se entrena con una línea: `.fit()` (ajustar). Y se usa con otra: `.predict()` (predecir).

In [ ]:
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_ent, y_ent)                    # aprende con los datos de entrenamiento

pred_lineal = modelo_lineal.predict(X_prueba)      # predice mezclas que NO vio

mae_lineal = mean_absolute_error(y_prueba, pred_lineal)
r2_lineal = r2_score(y_prueba, pred_lineal)
print(f"MAE (error promedio): {mae_lineal:.1f} MPa")
print(f"R² (0 = malo, 1 = perfecto): {r2_lineal:.2f}")

**Cómo leer las métricas**

- **MAE** (error absoluto medio): en promedio, el modelo se equivoca por esta cantidad de MPa. **Menor es mejor** y está en unidades de ingeniería, fáciles de interpretar.
- **R²**: qué fracción de la variación de la resistencia logra explicar el modelo. Cercano a 1 es mejor; cercano a 0 significa que casi no mejora a "predecir siempre el promedio".

In [ ]:
def grafica_real_vs_predicho(y_real, y_pred, titulo):
    plt.scatter(y_real, y_pred, s=10, alpha=0.6)
    limites = [min(y_real.min(), y_pred.min()), max(y_real.max(), y_pred.max())]
    plt.plot(limites, limites, color="red", linestyle="--", label="Predicción perfecta")
    plt.xlabel("Resistencia real (MPa)")
    plt.ylabel("Resistencia predicha (MPa)")
    plt.title(titulo)
    plt.legend()
    plt.show()


grafica_real_vs_predicho(y_prueba, pred_lineal, "Regresión lineal: real vs predicho")

Si el modelo fuera perfecto, todos los puntos caerían **sobre la línea roja**. Mientras más dispersos, peor la predicción.

¿Ves zonas donde el modelo se equivoca más? Una regresión lineal solo sabe dibujar rectas; la resistencia del concreto con la edad **no es una recta**. ¿Podemos hacerlo mejor?

## 7 · Segundo modelo: bosque aleatorio

Un **bosque aleatorio** (*random forest*) es un conjunto de muchos **árboles de decisión**: cada árbol hace preguntas del tipo *"¿la edad es mayor a 20 días? ¿el cemento es mayor a 300 kg?"* y llega a una predicción. El bosque **promedia las respuestas de cientos de árboles**. No asume que todo sea una recta, así que captura relaciones curvas.

Se usa **exactamente igual** que el modelo anterior: `.fit()` y `.predict()`.

In [ ]:
bosque = RandomForestRegressor(n_estimators=200, random_state=42)
bosque.fit(X_ent, y_ent)

pred_bosque = bosque.predict(X_prueba)

mae_bosque = mean_absolute_error(y_prueba, pred_bosque)
r2_bosque = r2_score(y_prueba, pred_bosque)
print(f"MAE (error promedio): {mae_bosque:.1f} MPa")
print(f"R² (0 = malo, 1 = perfecto): {r2_bosque:.2f}")

In [ ]:
grafica_real_vs_predicho(y_prueba, pred_bosque, "Bosque aleatorio: real vs predicho")

In [ ]:
comparacion = pd.DataFrame({
    "Modelo": ["Regresión lineal", "Bosque aleatorio"],
    "MAE prueba (MPa)": [mae_lineal, mae_bosque],
    "R² entrenamiento": [r2_score(y_ent, modelo_lineal.predict(X_ent)), r2_score(y_ent, bosque.predict(X_ent))],
    "R² prueba": [r2_lineal, r2_bosque],
}).round(2)
comparacion

**Discute:**

1. ¿Qué modelo se equivoca menos con datos nuevos?
2. Compara **R² de entrenamiento** con **R² de prueba** del bosque. Si en entrenamiento es mucho más alto, el modelo está **sobreajustado**: se aprendió "de memoria" parte de los ejemplos. Por eso evaluamos siempre con datos que no vio.

## 8 · ¿Qué variables usa el modelo para decidir?

In [ ]:
importancias = pd.Series(bosque.feature_importances_, index=X.columns).sort_values()
importancias.plot(kind="barh", color="#55A868")
plt.xlabel("Importancia relativa")
plt.title("Variables más usadas por el bosque aleatorio")
plt.show()
print(importancias.sort_values(ascending=False).round(2))

Esta gráfica indica **qué variables ayudan más al modelo a predecir**, no qué variables *causan* la resistencia. Aun así, es una pista valiosa: si el cemento y la edad encabezan la lista, coincide con lo que sabe la ingeniería civil. **Cuando el modelo contradice lo que sabe la ingeniería, desconfía primero del modelo.**

¿Coincide con la hipótesis que escribiste en la sección 4?

## 9 · De la predicción a la decisión

Un número de resistencia no es una decisión. La decisión es **aprobar o rechazar** una mezcla. Supongamos que la especificación exige **al menos 30 MPa** (valor ilustrativo para el ejercicio).

Hay dos formas de equivocarse, y **no cuestan lo mismo**:

| Error | Qué pasó | Consecuencia |
|---|---|---|
| **Falso aprobado** | El modelo dijo "cumple", pero **no** cumple | Riesgo estructural, mala calidad. **Es el error caro.** |
| **Falso rechazado** | El modelo dijo "no cumple", pero sí cumplía | Desperdicio: más cemento, más ensayos, más costo. |

Para protegernos del error caro usamos un **margen de seguridad**: solo aprobamos si la predicción supera el umbral **más** el error típico del modelo.

In [ ]:
UMBRAL = 30.0            # MPa exigidos (valor ilustrativo)
margen = mae_bosque      # margen de seguridad = error promedio del modelo

cumple_real = (y_prueba >= UMBRAL)                    # lo que realmente pasó
aprueba_simple = (pred_bosque >= UMBRAL)              # regla 1: sin margen
aprueba_seguro = (pred_bosque - margen >= UMBRAL)     # regla 2: con margen

def cuenta_errores(aprueba, nombre):
    falsos_aprobados = int(((aprueba) & (~cumple_real)).sum())
    falsos_rechazados = int(((~aprueba) & (cumple_real)).sum())
    return {"Regla": nombre, "Falsos aprobados (caro)": falsos_aprobados,
            "Falsos rechazados (desperdicio)": falsos_rechazados}

pd.DataFrame([cuenta_errores(aprueba_simple, "Sin margen"),
              cuenta_errores(aprueba_seguro, "Con margen")])

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for eje, aprueba, titulo in zip(ejes, [aprueba_simple, aprueba_seguro], ["Sin margen", "Con margen"]):
    cm = confusion_matrix(cumple_real, aprueba)
    ConfusionMatrixDisplay(cm, display_labels=["No cumple", "Cumple"]).plot(ax=eje, colorbar=False)
    eje.set_xlabel("Decisión del modelo")
    eje.set_ylabel("Realidad")
    eje.set_title(titulo)
    eje.grid(False)
plt.tight_layout()
plt.show()

Observa el compromiso: el margen **reduce los falsos aprobados** (lo peligroso) a cambio de **más falsos rechazados** (lo costoso). No hay regla perfecta: hay una **decisión de ingeniería** sobre cuánto riesgo se acepta. El modelo no la toma por ti.

### Aplicarlo a tres mezclas candidatas

In [ ]:
candidatas = pd.DataFrame([
    # cemento, escoria, ceniza, agua, superplast., grueso, fino, edad
    [300,   0,   0, 185,  5, 1000, 800, 28],   # Mezcla A: convencional
    [250, 100,   0, 190,  8,  980, 780, 28],   # Mezcla B: con escoria de alto horno
    [220,   0, 100, 175, 10, 1010, 770, 28],   # Mezcla C: con ceniza volante
], columns=X.columns, index=["Mezcla A", "Mezcla B", "Mezcla C"])

resultado = candidatas.copy()
resultado["resistencia_predicha"] = bosque.predict(candidatas).round(1)
resultado["predicha_menos_margen"] = (resultado["resistencia_predicha"] - margen).round(1)
resultado["decisión"] = np.where(resultado["predicha_menos_margen"] >= UMBRAL, "APROBAR", "REVISAR / RECHAZAR")

print(f"Umbral: {UMBRAL} MPa · margen de seguridad: {margen:.1f} MPa")
resultado[["resistencia_predicha", "predicha_menos_margen", "decisión"]]

### Límite importante: no extrapolar

Un modelo solo conoce lo que vio. Si le pides una mezcla **fuera del rango de los datos de entrenamiento**, responde igual, pero **sin respaldo**. Esta función avisa cuando una mezcla se sale del rango conocido.

In [ ]:
rango = X_ent.agg(["min", "max"])

def revisa_rango(mezcla):
    fuera = [c for c in X.columns if not (rango.loc["min", c] <= mezcla[c] <= rango.loc["max", c])]
    return "Dentro del rango" if not fuera else "FUERA de rango: " + ", ".join(fuera)

for nombre, fila in candidatas.iterrows():
    print(nombre, "→", revisa_rango(fila))

## 10 · Tu turno

Resuelve en parejas. Las soluciones están al final, **no las mires antes de intentarlo**.

1. **Edad.** Cambia la edad de las tres mezclas candidatas de 28 a **7 días**. ¿Cambia la decisión? ¿Tiene sentido físico?
2. **Variabilidad.** Cambia `random_state=42` por otro número (por ejemplo 7) en la sección 5 y vuelve a ejecutar desde ahí. ¿Cambian el MAE y el R²? ¿Qué te dice eso sobre confiar en un solo resultado?
3. **Margen.** Usa un margen de **2 × MAE**. ¿Cuántos falsos aprobados quedan? ¿A qué precio?
4. **Reto.** ¿Cuál es el **menor contenido de cemento** con el que la Mezcla A todavía se aprueba (con margen) a 28 días? Pista: prueba valores de cemento con un ciclo `for`. Menos cemento significa menos costo y menos CO₂.
5. **Reflexión.** Escribe 3 líneas: ¿aprobarías la Mezcla B sin hacer ningún ensayo de laboratorio? ¿Qué información adicional pedirías antes de decidir?

## Conclusiones

- El ML es una herramienta para **estimar con un error conocido**; la decisión sigue siendo del ingeniero.
- **Siempre se evalúa con datos que el modelo no vio.**
- El **error caro** y el **error barato** son distintos: el umbral y el margen son decisiones de ingeniería, no de matemáticas.
- Un modelo **no debe usarse fuera del rango** de sus datos de entrenamiento.
- Correlación e importancia de variables **no son causalidad**.

**Siguiente paso:** el **Workbook 2** repite el flujo con un problema de *clasificación* (aprobar o rechazar un lote de vino por sus análisis fisicoquímicos).

*Datos: Yeh, I.-C. (1998). Concrete Compressive Strength [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5PK67 (CC BY 4.0).*

---
## Soluciones (consultar después de intentarlo)

In [ ]:
# Solución 1 · Edad a 7 días
c7 = candidatas.copy()
c7["edad_dias"] = 7
r7 = c7.copy()
r7["resistencia_predicha"] = bosque.predict(c7).round(1)
r7["predicha_menos_margen"] = (r7["resistencia_predicha"] - margen).round(1)
r7["decisión"] = np.where(r7["predicha_menos_margen"] >= UMBRAL, "APROBAR", "REVISAR / RECHAZAR")
print("Con edad = 7 días:")
r7[["resistencia_predicha", "predicha_menos_margen", "decisión"]]

In [ ]:
# Solución 2 · Variabilidad con otra semilla
resultados = []
for semilla in [1, 7, 42, 99]:
    Xe, Xp, ye, yp = train_test_split(X, y, test_size=0.2, random_state=semilla)
    m = RandomForestRegressor(n_estimators=100, random_state=semilla).fit(Xe, ye)
    p = m.predict(Xp)
    resultados.append({"semilla": semilla, "MAE": round(mean_absolute_error(yp, p), 2), "R²": round(r2_score(yp, p), 3)})
pd.DataFrame(resultados)
# El resultado cambia un poco con cada partición: por eso conviene repetir y no confiar en un solo número.

In [ ]:
# Solución 3 · Margen de 2 × MAE
aprueba_doble = (pred_bosque - 2 * mae_bosque >= UMBRAL)
pd.DataFrame([cuenta_errores(aprueba_simple, "Sin margen"),
              cuenta_errores(aprueba_seguro, "Margen 1 × MAE"),
              cuenta_errores(aprueba_doble, "Margen 2 × MAE")])

In [ ]:
# Solución 4 · Menor cemento con el que la Mezcla A aún se aprueba (con margen)
base = candidatas.loc["Mezcla A"].copy()
encontrado = None
for cem in range(200, 451, 10):
    prueba = base.copy()
    prueba["cemento"] = cem
    fila = pd.DataFrame([prueba], columns=X.columns)
    pred = bosque.predict(fila)[0]
    if pred - margen >= UMBRAL:
        encontrado = (cem, round(pred, 1))
        break

if encontrado:
    print(f"Con {encontrado[0]} kg/m³ de cemento la predicción es {encontrado[1]} MPa y se aprueba con margen.")
else:
    print("Ninguna cantidad probada alcanza el umbral con margen.")
# Revisa también que el valor esté dentro del rango de entrenamiento antes de confiar en el resultado.

**Solución 5 (orientativa):** no lo aprobaría sin ensayo si la predicción con margen queda cerca del umbral, si algún valor de la mezcla está en el borde del rango de entrenamiento, o si el material (tipo de cemento, escoria, agregados locales) difiere del de los datos. Pediría: ensayos a 7 y 28 días de una tanda piloto, caracterización de los materiales locales y el historial de la planta. El modelo **prioriza qué ensayar**; no reemplaza el ensayo final.